In [ ]:
import numpy as np
import pandas as pd
import os
import numpy as np
import pandas as pd
import os
from pathlib import Path
import re
import pandas as pd
from collections import Counter

### PLOTING RESULTS FROM CATEGORIZATION ANALYSIS

In [ ]:
dfs_3b = []
dfs_7b = []
path_3b = "/Users/anonymous_user/Downloads/experiment_results/exp3_categorization_results/starcoder/categorization_deepseek_3b"
path_7b = "/Users/anonymous_user/Downloads/experiment_results/exp3_categorization_results/starcoder/categorization_deepseek_7b"

In [ ]:
dfs_llama_reg = []
paths_llama_reg = "/Users/anonymous_user/Downloads/experiment_results/exp3_categorization_results/LLaMa/reg_llama_categorization"

In [ ]:
### CLEAN UP, remove categories with less than 5 samples

def drop_small_categories(
    df: pd.DataFrame,
    category_col: str = "category_better_prompt_deepseek",
    min_samples: int = 35
) -> pd.DataFrame:
    """
    Return a copy of `df` with categories in `category_col` that occur
    fewer than `min_samples` times dropped.

    Parameters
    ----------
    df : pd.DataFrame
    category_col : str
        The name of the column holding your categorical labels.
    min_samples : int
        The minimum number of rows a category must have to be kept.

    Returns
    -------
    filtered_df : pd.DataFrame
        A new DataFrame containing only rows whose category frequency
        is >= min_samples.
    """
    # 1. compute counts
    counts = df[category_col].value_counts()

    # 2. find categories to keep
    keep = counts[counts >= min_samples].index

    # 3. filter and return a copy
    return df[df[category_col].isin(keep)].copy()


In [ ]:
for file in os.listdir(path_3b):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(path_3b, file))
        dfs_3b.append(df)
df_3b = pd.concat(dfs_3b, ignore_index=True)
for file in os.listdir(path_7b):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(path_7b, file))
        dfs_7b.append(df)
df_7b = pd.concat(dfs_7b, ignore_index=True)
for file in os.listdir(paths_llama_reg):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(paths_llama_reg, file))
        dfs_llama_reg.append(df)
df_llama_reg = pd.concat(dfs_llama_reg, ignore_index=True)

In [ ]:
df_3b_cleaned = drop_small_categories(df_3b, category_col="category_better_prompt_deepseek", min_samples=5)
df_7b_cleaned = drop_small_categories(df_7b, category_col="category_better_prompt_deepseek", min_samples=35)

In [ ]:
df_llama_reg_cleaned = drop_small_categories(df_llama_reg, category_col="category_better_prompt_deepseek", min_samples=35)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, MultipleLocator

def plot_extractability_by_category(
    df: pd.DataFrame,
    category_col: str = "category_better_prompt_deepseek",
    extractable_col: str = "test_bleu_3b",
    extractable_value=1,
    colors=("steelblue", "orange"),
    figsize=(7, 4.25),                 # ← match attack-results size
    sort_by_extractable: bool = True,
    title: str = "Extractability by Category",
    *,
    font_size: int = 10                # journal style fonts (base 10, title 12, ticks 9)
):
    """
    Stacked bar chart of extractability by category with journal-style typography
    and percentage y-axis, sized to match the attack-results figures.
    """
    # 1) Flag extractable
    df = df.copy()
    df["_is_ext"] = (df[extractable_col] == extractable_value).astype(int)

    # 2) Counts & proportions
    counts = (
        df.groupby([category_col, "_is_ext"])
          .size()
          .unstack(fill_value=0)
          .rename(columns={0: "Non-extractable", 1: "Extractable"})
    )
    props = counts.div(counts.sum(axis=1), axis=0)

    # 3) Optional sort by % extractable
    if sort_by_extractable:
        order = props["Extractable"].sort_values(ascending=False).index
        counts = counts.reindex(order)
        props  = props.reindex(order)

    # --- Typography (journal style) ---
    rc = {
        "font.family": "serif",
        "font.size": font_size,
        "axes.titlesize": font_size + 2,  # 12
        "axes.labelsize": font_size,      # 10
        "xtick.labelsize": font_size - 1, # 9
        "ytick.labelsize": font_size - 1, # 9
        "legend.fontsize": font_size - 1, # 9
        "figure.dpi": 300,                # crisp like the attack plots
    }

    with plt.rc_context(rc):
        # 4) Plot stacks
        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
        bottom = None
        for i, label in enumerate(["Non-extractable", "Extractable"]):
            ax.bar(
                counts.index,
                props[label],
                bottom=bottom,
                color=colors[i],
                label=label
            )
            bottom = props[label] if bottom is None else bottom + props[label]

        # 5) Annotate counts & percents
        for idx, cat in enumerate(counts.index):
            cum = 0
            for label in ["Non-extractable", "Extractable"]:
                cnt = counts.loc[cat, label]
                pct = props.loc[cat, label]
                y = cum + pct/2
                txt_color = "black" if label == "Extractable" else ("white" if pct > 0.2 else "black")
                ax.text(
                    idx, y,
                    f"{cnt}\n({pct:.0%})",
                    ha="center", va="center",
                    color=txt_color,
                    fontsize=font_size - 1
                )
                cum += pct

        # 6) Axes & title
        ax.set_ylim(0, 1)
        ax.set_ylabel("Percentage")
        ax.set_xlabel("Category")
        ax.set_title(title)

        # Percentage y-axis
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        ax.yaxis.set_major_locator(MultipleLocator(0.20))  # 0,20,...,100%
        ax.yaxis.set_minor_locator(MultipleLocator(0.10))

        # 7) Legend: Extractable on top, no title
        handles, labels = ax.get_legend_handles_labels()
        order = [labels.index("Extractable"), labels.index("Non-extractable")]
        ax.legend([handles[i] for i in order], [labels[i] for i in order], loc="upper left", title=None)

        # 8) Layout
        ax.yaxis.grid(False)
        plt.xticks(rotation=0)
        plt.show()


In [ ]:
plot_extractability_by_category(
    df_3b_cleaned,
    category_col="category_better_prompt_deepseek",
    extractable_col="test_bleu_3b",
    extractable_value=1,
    colors=("steelblue", "orange"),
    figsize=(10, 6),
    title = "Memorization Rate by Category for StarCoder 3B"
)

In [ ]:
plot_extractability_by_category(
    df_7b_cleaned,
    category_col="category_better_prompt_deepseek",
    extractable_col="test_bleu_7b",
    extractable_value=1,
    colors=("steelblue", "orange"),
    figsize=(10, 6),
    title = "Memorization Rate by Category for StarCoder 7B"
)

In [ ]:
df_llama_reg_cleaned.head()

In [ ]:
plot_extractability_by_category(
    df_llama_reg_cleaned,
    category_col="category_better_prompt_deepseek",
    extractable_col="extraction_BLEU_LM",
    extractable_value=1,
    colors=("steelblue", "orange"),
    figsize=(10, 6),
    title = "Memorization Rate by Category for Llama3 8B"
)